# MAGNet — Inference Demo

Generate multi-person motion with DFOT and visualize results in 3D.

**Pipeline:** Choose a config &rarr; Run inference &rarr; Visualize

| Config | Task |
|--------|------|
| `dd100.yaml` | Joint multi-person generation |
| `dd100_partner_prediction.yaml` | Predict one person given the other |
| `dd100_turn_taking.yaml` | Asynchronous leader-follower generation |
| `dd100_motion_control.yaml` | Generate person B conditioned on A's motion |
| `dd100_partner_inpainting.yaml` | Inpaint one person given the other |
| `dd100_inbetweening.yaml` | Fill motion between keyframes |

## 1. Select Config

Set `CONFIG` to one of the YAML files listed above.

In [1]:
CONFIG = "configs/inference/dfot/dd100.yaml"

In [2]:
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG)
print(f"Task:       {cfg.sampling_cfg.sampling_task}")
print(f"Checkpoint: {cfg.checkpoint_dir}")
print(f"Sequences:  {cfg.inference_data_list}")
print(f"Samples:    {cfg.sampling_cfg.sampling_num}")
print(f"Frames:     {cfg.sampling_cfg.sampling_seq_len}")

Task:       joint
Checkpoint: ./checkpoints/dfot/magnet_dd100/v0
Sequences:  ['Jive_005_001', 'Samba_010_004', 'Waltz_005_002']
Samples:    10
Frames:     200


## 2. Run DFOT Inference

This calls the inference script which loads the model, generates motion samples, and saves `.npz` files.

In [3]:
!bash scripts/run_dfot_inference.sh {CONFIG}

DFOTNetwork Cond
Loading model from checkpoints/vqvae/magnet_dd100/v0/checkpoints_100000/model.safetensors
Mode: Relative Canonical Cond
test_data file list: dict_keys(['Jive_005_001', 'Samba_010_004', 'Waltz_005_002'])
Jive_005_001
context_seq_len: 32
Sampling:   0%|                                         | 0/500 [00:00<?, ?it/s]cfg_scale_dict: {'clean': 1.0}
[SmoothingGuidance] step=0/500 grad_norm=0.396621 weight=2.0
[FootSkatingGuidance] step=0/500 grad_norm=0.154433 weight=2.0
Sampling:  20%|██████▎                         | 99/500 [00:03<00:16, 23.96it/s][SmoothingGuidance] step=100/500 grad_norm=0.417639 weight=2.0
[FootSkatingGuidance] step=100/500 grad_norm=0.186599 weight=2.0
Sampling:  40%|████████████▎                  | 198/500 [00:07<00:12, 23.47it/s][SmoothingGuidance] step=200/500 grad_norm=0.406479 weight=2.0
[FootSkatingGuidance] step=200/500 grad_norm=0.166056 weight=2.0
Sampling:  60%|██████████████████▌            | 300/500 [00:11<00:08, 23.25it/s][SmoothingGuidan

## 3. Locate Output Directory

Find the latest inference output (the script timestamps each run).

In [4]:
from pathlib import Path

output_base = Path(cfg.output_dir) / Path(cfg.checkpoint_dir).parents[0].name
output_dirs = sorted(output_base.glob("*"), key=lambda p: p.stat().st_mtime)
OUTPUT_DIR = str(output_dirs[-1])

print(f"Output: {OUTPUT_DIR}")
print(f"Files:")
for f in sorted(Path(OUTPUT_DIR).glob("*.npz")):
    print(f"  {f.name}")

Output: outputs/dfot/magnet_dd100/500K_test_joint_20260318_175256
Files:
  Jive_005_001.npz
  Samba_010_004.npz
  Waltz_005_002.npz


## 4. Visualize with Viser

Launch an interactive 3D viewer. Open the printed share URL in your browser.

**Interrupt the kernel** to stop the viewer and continue.

In [5]:
import sys
import viser

viz_dir = str(Path("libs/viz").resolve())
if viz_dir not in sys.path:
    sys.path.insert(0, viz_dir)

from libs.viz.visualizer import load_visualizer, visualize

server = viser.ViserServer(port=8084)
share_url = server.request_share_url()
print(f"Share URL: {share_url}")

try:
    visualize(server, data_root_dir=Path(OUTPUT_DIR))
except KeyboardInterrupt:
    print("Viewer stopped.")

╭────── viser (listening *:8084) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8084   │
│   Websocket │ ws://localhost:8084     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://tanh-unbiased.share.viser.studio

Share URL: https://tanh-unbiased.share.viser.studio


(viser) Connection opened (0, 1 total), 63 persistent messages

task mode: joint
updating smpl data
updated smpl data
task mode: joint
updating smpl data
updated smpl data
task mode: joint
updating smpl data
updated smpl data


(viser) Connection closed (0, 0 total)

Viewer stopped.


(viser) Disconnected from share URL